**NOTE:** Some variable names differ from those used in the main text, 
reflecting terminology changes made during the course of the study:
- 'fold' -> 'lineage'
- 'NONE' -> 'base seed'
- 'ALL' -> 'full network'

Each enzyme-gated network expansion run records iterations of network expansion as follows:
- cumiter: Tracks all individual expansion steps, including both lineage addition steps and the resulting standard network expansion steps
- folditer: Groups each lineage addition and the subsequent network expansion into a single unit; corresponds to a metabolic stage

In [ ]:
import networkExpansionPy.folds as nf
import pandas as pd
import pickle
from collections import Counter
from pathlib import PurePath, Path

# seed = sys.argv[1]
# random.seed(seed)
asset_path = nf.asset_path

# for Standard
METABOLISM_PATH = PurePath(asset_path, "metabolic_networks","metabolism.v8.01May2023.pkl") # path to metabolism object pickle
RN2RULES_PATH = PurePath(asset_path,"rn2fold","rn2rules.20230224.pkl") # path to rn2rules object pickle
SEED_CPDS_PATH = PurePath(asset_path, "compounds", "seeds.Goldford2022.csv") # path to seed compounds csv

# for Enzyme-gated
ALGORITHM = "no_look_ahead_rules"
WRITE = True # write result to disk
WRITE_TMP = False # write after each iteration
CUSTOM_WRITE_PATH = None # if writing result, custom path to write to
STR_TO_APPEND_TO_FNAME = None # if writing result, str to append to filename
ORDERED_OUTCOME = False # ignore random seed and always choose folds based on sort order
IGNORE_REACTION_VERSIONS = True # when maximizing for reactions, don't count versioned reactions

## Metabolism
metabolism = pd.read_pickle(METABOLISM_PATH)
# remove reactions that produce H2O2 before O2
H2O2_rns = ['R00017_v1', 'R03532_v1', 'R09507_v1', 'R09740_v1', 'R09741_v1', 'R11522', 'R12455', 'R12454']
condition = ((metabolism.network['rn'].isin(H2O2_rns)) & (metabolism.network['direction'] == 'reverse'))
metabolism.network = metabolism.network[~condition]
assert 'R00017_v1' not in list(metabolism.network['rn'])

## Compound ablation
# metabolism.pruneReactionsFromMetabolite(['C00007'])
# metabolism.pruneReactionsFromMetabolite(['Z00032'])
# metabolism.pruneReactionsFromMetabolite(['C00006'])

## FoldRules
rn2rules = pd.read_pickle(RN2RULES_PATH)
foldrules = nf.FoldRules.from_rn2rules(rn2rules)

## Modify seeds with AA and GATP_rns
aa_cids = set(["C00037",
    "C00041",
    "C00065",
    "C00188",
    "C00183",
    "C00407",
    "C00123",
    "C00148",
    "C00049",
    "C00025"])

# 70 seed cpds
# aa_cids = set([])

GATP_rns = {'R00200_gATP_v1',
    'R00200_gATP_v2',
    'R00430_gGTP_v1',
    'R00430_gGTP_v2',
    'R01523_gATP_v1',
    'R04144_gATP_v1',
    'R04208_gATP',
    'R04463_gATP',
    'R04591_gATP_v1',
    'R06836_gATP',
    'R06974_gATP',
    'R06975_gATP_v1'}


## Seed
seed = nf.Params(
    rns = set(metabolism.network["rn"]) - set(rn2rules) | GATP_rns,  # start with enzyme independent reactions + GATP_rns
    cpds = set((pd.read_csv(SEED_CPDS_PATH)["ID"])) | aa_cids,
    folds = set(['spontaneous'])
)

## Seed with pre-expansion
# preALL = pd.read_pickle('../data/pre-expansion_seed_cpds/pre-expansion_seed_cpds_ALL.pkl')
# seed = nf.Params(
#     rns = set(metabolism.network["rn"]) - set(rn2rules) | GATP_rns,  # start with enzyme independent reactions + GATP_rns
#     cpds = set(preALL) | aa_cids,
#     folds = set(['spontaneous'])
# )

# ## Inititalize fold metabolism
fm = nf.FoldMetabolism(metabolism, foldrules, seed)

# ## run Enzyme-gated expansion
# result = fm.rule_order(algorithm=ALGORITHM, write=WRITE, write_tmp=WRITE_TMP, path=CUSTOM_WRITE_PATH, str_to_append_to_fname=STR_TO_APPEND_TO_FNAME, ordered_outcome=ORDERED_OUTCOME, ignore_reaction_versions=IGNORE_REACTION_VERSIONS)

# save result object

In [ ]:
# with open('../data/assets/vanilla_result_object.pkl', 'wb') as file:
#     pickle.dump(fm, file)

# len(fm.seed.cpds)

# making 'rns_unreached_wo_cofactor' pkl files

In [ ]:
full_fm = fm

In [ ]:
len(full_fm.scope.rns)

In [ ]:
len(fm.scope.cpds), len(fm.scope.rns)

In [ ]:
len(full_fm.scope.rns - fm.scope.rns)

In [ ]:
full_rns_scope = full_fm.scope.rns

In [ ]:
print(len(full_rns_scope), len(fm.scope.rns))

my_set = full_rns_scope - fm.scope.rns
print(len(my_set))

In [ ]:
# with open('../data/rns_unreached_wo/rns_unreached_wo_O2.pkl', 'wb') as file:
#     pickle.dump(my_set, file)

# get seed sets for pre-expansion

In [ ]:
preexp_cpds = ['Z00035', 'C00002', 'C00004', 'C00019', 'C00010', 'C00016', 'Z00047', 'Z00009']

cpd2seeds = {}
cpd2seeds['NONE'] = fm.seed.cpds

for cpd in preexp_cpds:
    seeds = []
    cpditer = fm.scope.cpd_iteration_dict[cpd]
    for c, i in fm.scope.cpd_iteration_dict.items():
        if i <= cpditer:
            seeds.append(c)
    cpd2seeds[cpd] = seeds

cpd2seeds['ALL'] = fm.scope.cpds

In [ ]:
for c, seeds in cpd2seeds.items():
    print(c, len(seeds))

In [ ]:
# for cpd, seeds in cpd2seeds.items():
#     with open(f'../data/pre-expansion_seed_cpds/pre-expansion_seed_cpds_{cpd}.pkl', 'wb') as file:
#         pickle.dump(seeds, file)

### TableS1

In [ ]:
for c in fm.scope.cpds:
    if cpd2name.get(c, 'NONE') == 'NONE':
        print(c)

In [ ]:
cpd2name = csv2dict('../data/assets/cpd2name.csv')
pre2label = {'NONE': 'NONE', 'Z00035': 'PLP', 'C00002': 'ATP', 'C00004': 'NADH', 'C00019': 'SAM', 'C00010': 'CoA', 'C00016': 'FAD', 'Z00047': 'ThDP', 'Z00009': 'Cobalamin', 'ALL': 'ALL'}

df = pd.DataFrame(columns=['Pre-expansion', 'ID', 'Description'])
seen = set()

data = []
for pre in cpd2seeds.keys():
    l = len(cpd2seeds[pre])
    for c in sorted(cpd2seeds[pre]):
        if c not in seen:
            data.append({
                'Pre-expansion': f'{pre2label[pre]} ({l})',
                'ID': c,
                'Description': cpd2name.get(c, c)
            })
        seen.add(c)

df = pd.DataFrame(data)  # Convert list of dicts to DataFrame
df.head()

In [ ]:
# df.to_csv('TableS1.csv', index=False)

# Leave One Out (LOO) – lineage ablation

In [ ]:
import copy
import time

In [ ]:
all_xgroups = ['fold_independent', 'spontaneous', '2002', '1', '2487', '2006', '2003', '7518', '4126', '7542', '246', '218', '2007', '7525', '304', '12', '11', '109', '10', '210', '7515', '279', '7577', '3016', '281', '7528', '331', '301', '7572', '2011', '206', '325', '282', '7527', '805', '180', '7512', '7560', '7529', '3691', '7579', '62', '315', '7554', '70', '4002', '2485', '2004', '297', '205', '7580', '286', '208', '247', '7567', '7510', '3156', '2', '307', '212', '387', '7561', '5104', '7571', '5', '7516', '7517', '7574', '911', '611', '132', '283', '3651', '2005', '2498', '4081', '7573', '632', '103', '328', '64', '708', '302', '2484', '7520', '185', '221', '2492', '7543', '211', '810', '285', '2493', '7523', '4021', '298', '7541', '7552', '164', '3847', '149', '4020', '4045', '355', '4178', '108', '4018', '7587', '7546', '7507', '812', '101', '4019', '223', '3339', '702', '7601', '7602', '144', '3374', '278', '604', '7588', '602', '4952', '4953', '7524', '236', '131', '129', '159', '7581', '2486', '7531', '222', '213', '7504', '6094', '3692', '4004', '220', '330', '7558', '230', '314', '207', '65', '376', '276', '503', '7501', '244', '7522', '323', '7514', '107', '3294', '217', '187', '219', '239', '243', '3883', '7564', '266', '147', '321', '312', '296', '3687', '150', '3699', '5084', '868', '1001', '327', '3688', '375', '4995', '7589', '73', '133', '3371', '7568', '7509', '300', '277', '141', '7534', '3754', '2500', '1074', '102', '136', '316', '209', '139', '601', '4093', '177', '371', '873', '4279', '5038', '813', '5069', '7500', '3257', '172', '629', '232', '3052', '275', '6075', '7553', '3264', '231', '179', '189', '7539', '830', '377', '270', '313', '6051', '106', '3654', '7586', '258', '6058', '3117', '3292', '4114', '3086', '1077', '3599', '6113', '7550', '1137', '4262', '3005', '69', '7536', '7549', '214', '4', '3686', '148', '3896', '3076', '633', '4033', '4052', '3794', '2008', '1119', '4161', '319', '268', '5067', '9', '3447', '4029', '4049', '7584', '4011', '4237', '4111', '608', '7604', '7551', '525', '875', '880', '3858', '582', '4194', '374', '7513', '3740', '1143', '1055', '1114', '7595', '7562', '814', '197', '842', '650', '869', '5039', '7521', '623', '1144', '6096', '4022', '4044', '237', '5100', '4295', '7540', '309', '253', '303', '199', '3001', '4335', '3685', '324', '3009', '528', '4076', '876', '4017', '2496', '3697', '3892', '3997', '146', '262', '867', '914', '2012', '7', '3249', '158', '557', '66', '4294', '6166', '257', '3978', '620', '6174', '6', '4159', '4110', '235', '7556', '581', '3960', '154', '640', '5103', '3752', '3777', '3115', '4160', '196', '305', '3207', '881', '241', '284', '3321', '3323', '3322', '920', '378', '7544', '184', '4983', '4229', '4971', '4223', '590', '2010', '169', '806', '228', '192', '603', '306', '75', '3018', '865', '872', '3269', '7563', '3623', '4048', '4028', '7578', '3993', '3994', '3500', '3304', '4025', '4024', '4046', '4272', '3281', '3843', '3456', '3579', '4035', '4054', '4036']
len(all_xgroups)

import networkExpansionPy.folds as nf
import pandas as pd
from collections import Counter
from pathlib import PurePath, Path

for x in all_xgroups:
    xgroups = copy.copy(all_xgroups)
    xgroups.remove(x)

    start_time = time.time()

    # seed = sys.argv[1]
    # random.seed(seed)
    asset_path = nf.asset_path

    # for vanilla
    METABOLISM_PATH = PurePath(asset_path, "metabolic_networks","metabolism.v8.01May2023.pkl") # path to metabolism object pickle
    RN2RULES_PATH = PurePath(asset_path,"rn2fold","rn2rules.20230224.pkl") # path to rn2rules object pickle
    SEED_CPDS_PATH = PurePath(asset_path, "compounds", "seeds.Goldford2022.csv") # path to seed compounds csv

    # for Enzyme-gated
    ALGORITHM = "no_look_ahead_rules"
    WRITE = True # write result to disk
    WRITE_TMP = False # write after each iteration
    CUSTOM_WRITE_PATH = None # if writing result, custom path to write to
    STR_TO_APPEND_TO_FNAME = f'{x}' # if writing result, str to append to filename
    ORDERED_OUTCOME = False # ignore random seed and always choose folds based on sort order
    IGNORE_REACTION_VERSIONS = True # when maximizing for reactions, don't count versioned reactions

    ## Metabolism
    metabolism = pd.read_pickle(METABOLISM_PATH)
    # remove reactions that produce H2O2 before O2
    H2O2_rns = ['R00017_v1', 'R03532_v1', 'R09507_v1', 'R09740_v1', 'R09741_v1', 'R11522', 'R12455', 'R12454']
    condition = ((metabolism.network['rn'].isin(H2O2_rns)) & (metabolism.network['direction'] == 'reverse'))
    metabolism.network = metabolism.network[~condition]
    assert 'R00017_v1' not in list(metabolism.network['rn'])


    ## FoldRules
    rn2rules = pd.read_pickle(RN2RULES_PATH)
    foldrules = nf.FoldRules.from_rn2rules(rn2rules)

    ## Modify seeds with AA and GATP_rns
    aa_cids = set(["C00037",
        "C00041",
        "C00065",
        "C00188",
        "C00183",
        "C00407",
        "C00123",
        "C00148",
        "C00049",
        "C00025"]) 

    GATP_rns = {'R00200_gATP_v1',
        'R00200_gATP_v2',
        'R00430_gGTP_v1',
        'R00430_gGTP_v2',
        'R01523_gATP_v1',
        'R04144_gATP_v1',
        'R04208_gATP',
        'R04463_gATP',
        'R04591_gATP_v1',
        'R06836_gATP',
        'R06974_gATP',
        'R06975_gATP_v1'}

    ## Seed
    seed = nf.Params(
        rns = set(metabolism.network["rn"]) - set(rn2rules) | GATP_rns,  # start with enzyme independent reactions
        cpds = set((pd.read_csv(SEED_CPDS_PATH)["ID"])) | aa_cids,
        folds = set(xgroups)
    )

    ## Inititalize fold metabolism
    fm = nf.FoldMetabolism(metabolism, foldrules, seed)

    # run Enzyme-gated expansion
    result = fm.rule_order(algorithm=ALGORITHM, write=WRITE, write_tmp=WRITE_TMP, path=CUSTOM_WRITE_PATH, str_to_append_to_fname=STR_TO_APPEND_TO_FNAME, ordered_outcome=ORDERED_OUTCOME, ignore_reaction_versions=IGNORE_REACTION_VERSIONS)

    print("--- %s seconds ---" % (time.time() - start_time))